In [ ]:
import sys
import os
import time
import yaml
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
import pybullet as p

# ==========================================
# CONFIGURE WHICH STAGE TO WATCH:
STAGE_TO_WATCH = 0
# ==========================================

print(f"Presentation Mode Active: Loading Stage {STAGE_TO_WATCH}...")

# Load stage parameters from the YAML config file
config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]
reward_weights = config['rewards']

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
RUN_NAME = stage_config['run_name']

# Initialize the environment parametrically based on the selected stage
env = RoomDroneEnv(gui=True, num_obstacles=NUM_OBS, randomize_obstacles=RAND_OBS, randomize_coins=RAND_COINS, reward_weights=reward_weights)

# Locate the best_model.zip inside the specific stage's folder
model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'best_model'))
print(f"Loading Brain from: {model_path}.zip")

try:
    model = PPO.load(model_path, env=env)
except Exception as e:
    print(f"ERROR: Model for this stage is not trained yet or cannot be found!\nDetails: {e}")
    sys.exit()

obs, info = env.reset()
# Set a nice camera angle for the showcase
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] Simulation started. Runs until collision...")

while True:
    # The agent predicts the best action based on the observation
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action) 
    
    # Maintain real-time execution speed (240 Hz physics engine)
    time.sleep(1./240.) 
    
    if terminated or truncated:
        print("Episode finished. Loading new room...")
        time.sleep(1) # Brief pause before resetting
        obs, info = env.reset()